# Star schema: facts and dimensions

Practice notebook for the datafromzero SQL lesson **Star schema: facts and dimensions**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

The views live in the notebook's in-memory database. If the notebook restarts, run the setup cell and the `CREATE` cells again.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Declare the grain, then build the fact view

Grain: one row per order line.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW fact_sales AS
SELECT i.order_item_id, i.order_id,
       CAST(o.ordered_at AS DATE) AS order_date,
       o.store_id, o.customer_id, i.product_id,
       o.status AS order_status,
       i.quantity, i.discount_usd,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
""").show()

### Step 2. Check the grain

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines, COUNT(DISTINCT order_item_id) AS distinct_lines,
       ROUND(SUM(amount_usd), 2) AS sales
FROM fact_sales
""").show()

### Step 3. Build the dimensions

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW dim_store AS
SELECT store_id, store_name, city FROM stores
""").show()

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW dim_product AS
SELECT product_id, product_name, category FROM products
""").show()

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW dim_date AS
SELECT CAST(generate_series AS DATE) AS order_date,
       year(generate_series) AS year, month(generate_series) AS month,
       dayname(generate_series) AS weekday,
       isodow(generate_series) >= 6 AS is_weekend
FROM generate_series(TIMESTAMP '2026-01-01', TIMESTAMP '2026-06-30', INTERVAL 1 DAY)
""").show()

### Step 4. Check every key

The row count and the distinct key count must match.

In [ ]:
con.sql("""
SELECT COUNT(*) AS dim_rows, COUNT(DISTINCT product_id) AS distinct_keys FROM dim_product
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS dim_rows, COUNT(DISTINCT store_id) AS distinct_keys FROM dim_store
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS dim_rows, COUNT(DISTINCT order_date) AS distinct_keys FROM dim_date
""").show()

### Step 5. Ask a question through the star

In [ ]:
con.sql("""
SELECT p.category, d.is_weekend, ROUND(SUM(f.amount_usd), 2) AS sales
FROM fact_sales AS f
JOIN dim_product AS p ON p.product_id = f.product_id
JOIN dim_date AS d ON d.order_date = f.order_date
WHERE f.order_status = 'completed'
GROUP BY p.category, d.is_weekend
ORDER BY p.category, d.is_weekend
""").show()

### Step 6. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines, ROUND(SUM(f.amount_usd), 2) AS sales
FROM fact_sales AS f
JOIN dim_store AS s ON s.store_id = f.store_id
JOIN dim_product AS p ON p.product_id = f.product_id
JOIN dim_date AS d ON d.order_date = f.order_date
""").show()

In [ ]:
con.sql("""
WITH by_group AS (
SELECT p.category, d.is_weekend, ROUND(SUM(f.amount_usd), 2) AS sales
FROM fact_sales AS f
JOIN dim_product AS p ON p.product_id = f.product_id
JOIN dim_date AS d ON d.order_date = f.order_date
WHERE f.order_status = 'completed'
GROUP BY p.category, d.is_weekend
)
SELECT COUNT(*) AS groups, ROUND(SUM(sales), 2) AS sales FROM by_group
""").show()

## Exercise 1. Completed sales for each store in each month

Finish the query: the `FROM` and the joins are done for you. Add the `SELECT` (store name, month, rounded sum of `amount_usd`), the filter on completed orders, the `GROUP BY` and an `ORDER BY`.

In [ ]:
con.sql("""
SELECT *
-- replace this line with: the store name, the month, the rounded sum of amount_usd
FROM fact_sales AS f
JOIN dim_store AS s ON s.store_id = f.store_id
JOIN dim_date AS d ON d.order_date = f.order_date
LIMIT 3
-- then add the filter on completed orders, GROUP BY and ORDER BY (replace LIMIT)
""").show()

## Exercise 2. Completed sales for each loyalty tier

Before you run anything, write one sentence in the cell below: what the tiers must add up to. Then create `dim_customer` (`customer_id`, `loyalty_tier`, `city` from `customers`), join it to `fact_sales`, group by tier, and compare the sum of the tiers with your sentence. Repair the query if they differ.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Create dim_customer:

In [ ]:
# Sales by loyalty tier:

## Exercise 3. Spot the flaw

A colleague builds a product dimension of sold products only (first cell), joins the fact to it (second cell) and reports the total sales of all lines. Run both, then find out what went wrong. Hint: run the key check from step 4 on the new view (third cell).

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW dim_product_sold AS
SELECT p.product_id, p.product_name, p.category
FROM products AS p
JOIN order_items AS i ON i.product_id = p.product_id
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines, ROUND(SUM(f.amount_usd), 2) AS sales
FROM fact_sales AS f
JOIN dim_product_sold AS p ON p.product_id = f.product_id
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS dim_rows, COUNT(DISTINCT product_id) AS distinct_keys FROM dim_product_sold
""").show()

In [ ]:
# Your corrected query: